In [21]:
# Centralized inputs for this notebook.
# Edit prepare_mgp_config.py, or set MGP_DATA_ROOT,
# MGP_HUMANN_DATASET, MGP_EDGE_DATASET, and MGP_GRAPH_PREFIX before starting Jupyter.

import importlib.util
import sys
from pathlib import Path

_CWD = Path.cwd().resolve()
_CONFIG_CANDIDATES = [
    _CWD / "prepare_mgp_config.py",
    _CWD / "data_process" / "prepare_mgp_config.py",
    _CWD / "MGPAN" / "data_process" / "prepare_mgp_config.py",
]

for _config_path in _CONFIG_CANDIDATES:
    if _config_path.exists():
        _config_path = _config_path.resolve()
        break
else:
    searched = "\n".join(str(path) for path in _CONFIG_CANDIDATES)
    raise FileNotFoundError(
        f"Cannot locate prepare_mgp_config.py. Searched:\n{searched}"
    )

_config_dir = str(_config_path.parent)
if _config_dir not in sys.path:
    sys.path.insert(0, _config_dir)

sys.modules.pop("prepare_mgp_config", None)
_spec = importlib.util.spec_from_file_location("prepare_mgp_config", _config_path)
if _spec is None or _spec.loader is None:
    raise ImportError(f"Cannot load prepare_mgp_config.py from {_config_path}")
cfg = importlib.util.module_from_spec(_spec)
sys.modules["prepare_mgp_config"] = cfg
_spec.loader.exec_module(cfg)

print(f"Loaded config: {_config_path}")
cfg.print_config()


Loaded config: D:\coursework\研究生\code\MGPAN\data_process\prepare_mgp_config.py
PROJECT_ROOT: D:\coursework\研究生\code
PACKAGE_ROOT: D:\coursework\研究生\code\MGPAN
DATA_ROOT: D:\coursework\研究生\code\MGPAN\data
HUMAnN dataset: QinN_2014 -> D:\coursework\研究生\code\MGPAN\data\QinN_2014
Edge dataset: QinN_2014 -> D:\coursework\研究生\code\MGPAN\data\QinN_2014
P-P input: D:\coursework\研究生\code\MGPAN\data\QinN_2014\pathway_abundance_abundance.csv
M-M input: D:\coursework\研究生\code\MGPAN\data\QinN_2014\relative_abundance_abundance.csv
P-P output dir: D:\coursework\研究生\code\MGPAN\data\QinN_2014\pathway
M-M output dir: D:\coursework\研究生\code\MGPAN\data\QinN_2014\microbe
Metadata: D:\coursework\研究生\code\MGPAN\data\QinN_2014\relative_abundance_metadata.csv
Subject IDs: D:\coursework\研究生\code\MGPAN\data\QinN_2014\subject_ids.csv
Raw graph: D:\coursework\研究生\code\MGPAN\data\QinN_2014\QN_graphdata.bin
Raw graph metadata: D:\coursework\研究生\code\MGPAN\data\QinN_2014\QN_graphdata_meta.pkl
Pruned graph: D:\coursew

## 1. Load abundance tables and triplets


In [22]:
import os
import pandas as pd
import torch

cfg.validate_graph_inputs()
cfg.ensure_output_dirs()

microbe_edges = cfg.MM_OUTPUT_DIR
pathway_edges = cfg.PP_OUTPUT_DIR
mgp_edges = cfg.SAMPLE_PATH_TAX_GENE_WEIGHT_CSV
microbe_abundance_file = cfg.MICROBE_ABUNDANCE_CLEANED_CSV
pathway_abundance_file = cfg.PATHWAY_ABUNDANCE_CLEANED_CSV
gene_abundance_file = cfg.GENE_FAMILIES_CSV

microbe_abundance_df = pd.read_csv(microbe_abundance_file, index_col=0)
pathway_abundance_df = pd.read_csv(pathway_abundance_file, index_col=0)
gene_abundance_df = pd.read_csv(gene_abundance_file, index_col=0)


## 2. Count labels


In [23]:
def load_sample_labels(metadata_file, label_col="study_condition", id_col="sample_id"):
    """
    根据 metadata 文件提取样本标签。
    默认使用 'disease' 列作为标签，'NCBI_accession' 作为样本ID。
    """
    # 1. 读取
    meta = pd.read_csv(metadata_file)

    # 有些CSV首列是空名，但其实是index
    # if meta.columns[0] in ["Unnamed: 0", ""]:
    #     meta = meta.rename(columns={meta.columns[0]: "sample_id"})
    #     id_col = "sample_id"

    # 2. 检查列存在
    if id_col not in meta.columns:
        raise ValueError(f"找不到样本ID列：{id_col}")
    if label_col not in meta.columns:
        raise ValueError(f"找不到标签列：{label_col}")

    # 3. 删除空值
    meta = meta[[id_col, label_col]].dropna()

    # 4. 建立映射
    label_map = {sid: lab for sid, lab in zip(meta[id_col], meta[label_col])}

    # 5. 编码为数字标签（例如 Healthy→0, CD→1, UC→2）
    label_set = sorted(list(set(label_map.values())))
    label_to_id = {lab: i for i, lab in enumerate(label_set)}
    label_encoded = {sid: label_to_id[lab] for sid, lab in label_map.items()}

    print("标签类别映射：", label_to_id)

    return label_encoded, label_to_id
    
import os
import glob

def filter_existing_samples(sample_ids, microbe_edges_dir, pathway_edges_dir):
    valid_samples = []
    for sid in sample_ids:
        # 微生物边可能是 "_microbe_edges.csv" 或 "_MM_edges.csv"
        microbe_patterns = [
            f"{sid}_microbe_edges.csv",
            f"{sid}_MM_edges.csv",
        
        ]
        # 通路边可能是 "_pathway_edges.csv" 或 "_PP_edges.csv"
        pathway_patterns = [
            f"{sid}_pathway_edges.csv",
            f"{sid}_PP_edges.csv",
        ]

        # 检查是否存在任意匹配文件
        found_microbe = any(
            os.path.exists(os.path.join(microbe_edges_dir, p))
            for p in microbe_patterns
        )
        found_pathway = any(
            os.path.exists(os.path.join(pathway_edges_dir, p))
            for p in pathway_patterns
        )

        if found_microbe and found_pathway:
            valid_samples.append(sid)
    return valid_samples

def filter_labels(label_dict, sample_ids):
    """从标签字典中提取指定样本对应的标签张量"""
    return torch.tensor([label_dict[sid] for sid in sample_ids], dtype=torch.long)

## 2.1 Original dependency cell: create all_sample_ids and labels


In [24]:
metadata_file = cfg.METADATA_CSV

label_dict, label_to_id = load_sample_labels(
    metadata_file,
    label_col=cfg.LABEL_COL,
    id_col=cfg.SAMPLE_ID_COL,
)

# 全部样本 ID
all_sample_ids = list(label_dict.keys())

# ===============================
# 3. 检查输出
# ===============================
print(f"总样本数: {len(all_sample_ids)}")
print(all_sample_ids)

labels = torch.tensor(
    [label_dict[sid] for sid in all_sample_ids],
    dtype=torch.long
)

labels1 = torch.tensor([label_dict[k] for k in sorted(label_dict.keys())], dtype=torch.long)
print(f"总标签数：{len(labels)}")
print(labels)
print("labels.shape :", labels.shape)
print("labels1.shape:", labels1.shape)

same_tensor = torch.equal(labels, labels1)
print("labels 与 labels1 是否完全相同:", same_tensor)

num_0 = (labels == 0).sum().item()
num_1 = (labels == 1).sum().item()

print(num_0, num_1)


标签类别映射： {'cirrhosis': 0, 'control': 1}
总样本数: 237
['HD-1', 'HD-10', 'HD-11', 'HD-12', 'HD-13', 'HD-14', 'HD-15', 'HD-16', 'HD-17', 'HD-18', 'HD-19', 'HD-2', 'HD-20', 'HD-21', 'HD-22', 'HD-23', 'HD-24', 'HD-25', 'HD-26', 'HD-27', 'HD-28', 'HD-29', 'HD-3', 'HD-30', 'HD-31', 'HD-32', 'HD-33', 'HD-34', 'HD-35', 'HD-36', 'HD-37', 'HD-38', 'HD-39', 'HD-4', 'HD-40', 'HD-41', 'HD-42', 'HD-43', 'HD-44', 'HD-45', 'HD-46', 'HD-47', 'HD-48', 'HD-49', 'HD-5', 'HD-50', 'HD-51', 'HD-52', 'HD-53', 'HD-54', 'HD-55', 'HD-56', 'HD-57', 'HD-58', 'HD-59', 'HD-6', 'HD-60', 'HD-61', 'HD-62', 'HD-63', 'HD-64', 'HD-65', 'HD-66', 'HD-67', 'HD-68', 'HD-69', 'HD-7', 'HD-70', 'HD-71', 'HD-72', 'HD-73', 'HD-74', 'HD-75', 'HD-76', 'HD-77', 'HD-78', 'HD-79', 'HD-8', 'HD-80', 'HD-81', 'HD-82', 'HD-83', 'HD-9', 'HV-1', 'HV-10', 'HV-11', 'HV-12', 'HV-13', 'HV-14', 'HV-15', 'HV-16', 'HV-17', 'HV-18', 'HV-19', 'HV-2', 'HV-20', 'HV-21', 'HV-22', 'HV-23', 'HV-24', 'HV-25', 'HV-26', 'HV-27', 'HV-28', 'HV-29', 'HV-3', 'HV-30',

## 3. Save subject_id


In [25]:
import pandas as pd

meta = pd.read_csv(cfg.METADATA_CSV)
subject_ids = meta[cfg.SUBJECT_ID_COL]

subject_ids.to_csv(cfg.SUBJECT_IDS_CSV, index=False)


## 4. Build graph dataset


In [26]:
from graph_dataset import GraphDataset
import numpy as np
import dgl
import pickle

print("NumPy version:", np.__version__)
print("GraphDataset 继承自:", GraphDataset.__bases__)

cfg.validate_graph_inputs()

mgp_df = pd.read_csv(cfg.SAMPLE_PATH_TAX_GENE_WEIGHT_CSV)
mgp_df = mgp_df.dropna(
    subset=[
        "Sample",
        "pathway",
        "Taxonomy",
        "gene",
        "GeneAbundance",
        "PathwayAbundance",
        "TaxonomyAbundance",
        "weight_mg",
        "weight_gm",
        "weight_gp",
        "weight_pg",
    ]
)

mgp_by_sample = {
    sid: sub.reset_index(drop=True)
    for sid, sub in mgp_df.groupby("Sample")
}

print("mgp_by_sample 样本数量：", len(mgp_by_sample))

filtered_sample_ids = [
    sid for sid in all_sample_ids
    if sid in mgp_by_sample
]

filtered_indices = [
    i for i, sid in enumerate(all_sample_ids)
    if sid in mgp_by_sample
]

filtered_labels = labels[filtered_indices]

print(filtered_indices)
print(f"原始样本数量：{len(all_sample_ids)}, 过滤后：{len(filtered_sample_ids)}")
print("样本列表：", list(mgp_by_sample.keys()))

data = GraphDataset(
    sample_ids=filtered_sample_ids,
    labels=filtered_labels,
    microbe_edges_dir=str(cfg.MM_OUTPUT_DIR),
    pathway_edges_dir=str(cfg.PP_OUTPUT_DIR),
    mgp_by_sample=mgp_by_sample,
    microbe_abundance_df=microbe_abundance_df,
    pathway_abundance_df=pathway_abundance_df,
    gene_abundance_df=gene_abundance_df,
)

meta = [
    {
        "microbe": g.microbe_names,
        "gene": g.gene_names,
        "pathway": g.pathway_names,
    }
    for g in data.graphs
]

with open(cfg.GRAPH_RAW_META_PKL, "wb") as f:
    pickle.dump(meta, f)

dgl.save_graphs(
    str(cfg.GRAPH_RAW_BIN),
    data.graphs,
    {"labels": data.labels},
)

print(f"图数据已保存: {cfg.GRAPH_RAW_BIN}")
print(f"图元数据已保存: {cfg.GRAPH_RAW_META_PKL}")


NumPy version: 1.25.0
GraphDataset 继承自: (<class 'torch.utils.data.dataset.Dataset'>,)
mgp_by_sample 样本数量： 237
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199,

## 5. Filter samples without triplets


In [27]:
metadata_file = cfg.METADATA_CRC_CSV if cfg.METADATA_CRC_CSV.exists() else cfg.METADATA_CSV
meta = pd.read_csv(metadata_file)

meta[cfg.SAMPLE_ID_COL] = meta[cfg.SAMPLE_ID_COL].astype(str).str.strip()
meta[cfg.SUBJECT_ID_COL] = meta[cfg.SUBJECT_ID_COL].astype(str).str.strip()

sample_to_subject = dict(
    zip(meta[cfg.SAMPLE_ID_COL], meta[cfg.SUBJECT_ID_COL])
)

print("filtered_sample_ids 数量:", len(filtered_sample_ids))

filtered_subject_ids = [
    sample_to_subject[sid]
    for sid in filtered_sample_ids
    if sid in sample_to_subject
]

pd.Series(filtered_subject_ids, name=cfg.SUBJECT_ID_COL).to_csv(
    cfg.SUBJECT_IDS_CSV,
    index=False
)

print("✅ subject_id 已保存，样本数：", len(filtered_subject_ids))
print("✅ subject 数量：", len(set(filtered_subject_ids)))


filtered_sample_ids 数量: 237
✅ subject_id 已保存，样本数： 237
✅ subject 数量： 237


In [28]:
metadata_file = cfg.METADATA_CSV
meta = pd.read_csv(metadata_file)

filtered_subject_ids = meta[
    meta[cfg.SAMPLE_ID_COL].isin(mgp_by_sample.keys())
][[cfg.SUBJECT_ID_COL]]

num_filtered_samples = len(filtered_subject_ids)
print(f"过滤后的样本数量：{num_filtered_samples}")

filtered_subject_ids.to_csv(cfg.SUBJECT_IDS_CSV, index=False)


过滤后的样本数量：237


## 6. Remove nodes and edges by abundance


In [29]:
import dgl
import torch
import pickle
from tqdm import tqdm
from collections import defaultdict
import numpy as np
import pandas as pd

# ======================================================
# 1️⃣ 读取图和标签
# ======================================================
all_graphs, data_labels_dict = dgl.load_graphs(str(cfg.GRAPH_RAW_BIN))
all_labels = data_labels_dict["labels"]

with open(cfg.GRAPH_RAW_META_PKL, "rb") as f:
    all_meta = pickle.load(f)

print(f"原始样本数: {len(all_graphs)}")

# ======================================================
# 2️⃣ pruning 参数
# ======================================================
ABUNDANCE_THRESH = cfg.GRAPH_PRUNE_ABUNDANCE_THRESHOLDS
TARGET_TYPES = set(ABUNDANCE_THRESH.keys())
filtered_graphs = []
filtered_meta = []

node_count_stats = defaultdict(list)

# ======================================================
# 3️⃣ 逐图 pruning
# ======================================================
for idx, (g, meta) in enumerate(tqdm(zip(all_graphs, all_meta), total=len(all_graphs))):

    keep_nodes = {}
    meta_new = meta.copy()

    for ntype in g.ntypes:

        num_nodes = g.num_nodes(ntype)

        # ---------------------------------------------
        # gene / pathway：按各自阈值筛选
        # ---------------------------------------------
        if ntype in TARGET_TYPES:
            feat = g.nodes[ntype].data["feat"].float()

            abundance = feat[:, :3]
            importance = abundance.sum(dim=1)

            thresh = ABUNDANCE_THRESH[ntype]

            keep_idx = torch.nonzero(
                importance > thresh,
                as_tuple=False
            ).squeeze(1)

            # 防止空图
            if keep_idx.numel() == 0:
                keep_idx = torch.topk(
                    importance,
                    k=1,
                    largest=True
                ).indices

            keep_nodes[ntype] = keep_idx

            # 同步 meta
            if ntype in meta:
                meta_new[ntype] = [meta[ntype][i] for i in keep_idx.tolist()]

        # ---------------------------------------------
        # 其他节点类型：全部保留
        # ---------------------------------------------
        else:
            keep_nodes[ntype] = torch.arange(num_nodes)

        node_count_stats[ntype].append(len(keep_nodes[ntype]))

    g_new = dgl.node_subgraph(g, keep_nodes)

    filtered_graphs.append(g_new)
    filtered_meta.append(meta_new)

    if idx % 20 == 0:
        info = ", ".join(
            f"{ntype}: {len(keep_nodes[ntype])}"
            for ntype in keep_nodes
        )
        print(f"[Graph {idx}] {info}")


# ======================================================
# 4️⃣ 打印统计信息
# ======================================================
print("\n===== 节点数量统计（筛选后） =====")
for ntype, counts in node_count_stats.items():
    counts = np.array(counts)
    print(
        f"{ntype:10s} | "
        f"mean={counts.mean():6.1f} | "
        f"median={np.median(counts):6.1f} | "
        f"min={counts.min():4d} | "
        f"max={counts.max():4d}"
    )

# ======================================================
# 5️⃣ 重新计算边的权重
# ======================================================
def recompute_edge_weights_from_feat(g, EPS=1e-8):
    """
    在 pruning 后的单样本异构图 g 内，
    基于节点 feat 中的丰度信息，
    重新计算：MG, GM, GP, PG 四类边权
    """
    # =====================================================
    # 取出节点“丰度”
    # 假设 feat[:, 0] 是 abundance
    # =====================================================
    microbe_abund_all = g.nodes['microbe'].data['feat'][:, 0].float()
    gene_abund_all    = g.nodes['gene'].data['feat'][:, 0].float()
    pathway_abund_all = g.nodes['pathway'].data['feat'][:, 0].float()

    # =====================================================
    # MG: microbe -> gene
    #   每个 microbe 内部：按 GeneAbundance 归一化
    # =====================================================
    if 'MG' in g.etypes and g.num_edges('MG') > 0:
        src, dst = g.edges(etype='MG')  # microbe -> gene
        src = src.long()
        dst = dst.long()

        gene_abund = gene_abund_all[dst]

        df_mg = pd.DataFrame({
            'microbe': src.numpy(),
            'gene': dst.numpy(),
            'GeneAbundance': gene_abund.numpy()
        })

        df_mg['weight_mg'] = df_mg.groupby('microbe')['GeneAbundance'] \
                                  .transform(lambda x: x / (x.sum() + EPS))

        g.edges['MG'].data['weight'] = torch.tensor(
            df_mg['weight_mg'].values,
            dtype=torch.float32
        )

    # =====================================================
    # GM: gene -> microbe
    #   每个 gene 内部：按 TaxonomyAbundance 归一化
    # =====================================================
    if 'GM' in g.etypes and g.num_edges('GM') > 0:
        src, dst = g.edges(etype='GM')  # gene -> microbe
        src = src.long()
        dst = dst.long()

        microbe_abund = microbe_abund_all[dst]

        df_gm = pd.DataFrame({
            'gene': src.numpy(),
            'microbe': dst.numpy(),
            'TaxonomyAbundance': microbe_abund.numpy()
        })

        df_gm['weight_gm'] = df_gm.groupby('gene')['TaxonomyAbundance'] \
                                  .transform(lambda x: x / (x.sum() + EPS))

        g.edges['GM'].data['weight'] = torch.tensor(
            df_gm['weight_gm'].values,
            dtype=torch.float32
        )

    # =====================================================
    # GP: gene -> pathway
    #   每个 gene 内部：按 PathwayAbundance 归一化
    # =====================================================
    if 'GP' in g.etypes and g.num_edges('GP') > 0:
        src, dst = g.edges(etype='GP')  # gene -> pathway
        src = src.long()
        dst = dst.long()

        pathway_abund = pathway_abund_all[dst]

        df_gp = pd.DataFrame({
            'gene': src.numpy(),
            'pathway': dst.numpy(),
            'PathwayAbundance': pathway_abund.numpy()
        })

        df_gp['weight_gp'] = df_gp.groupby('gene')['PathwayAbundance'] \
                                  .transform(lambda x: x / (x.sum() + EPS))

        g.edges['GP'].data['weight'] = torch.tensor(
            df_gp['weight_gp'].values,
            dtype=torch.float32
        )

    # =====================================================
    # PG: pathway -> gene
    #   每个 pathway 内部：按 GeneAbundance 归一化
    # =====================================================
    if 'PG' in g.etypes and g.num_edges('PG') > 0:
        src, dst = g.edges(etype='PG')  # pathway -> gene
        src = src.long()
        dst = dst.long()

        gene_abund = gene_abund_all[dst]

        df_pg = pd.DataFrame({
            'pathway': src.numpy(),
            'gene': dst.numpy(),
            'GeneAbundance': gene_abund.numpy()
        })

        df_pg['weight_pg'] = df_pg.groupby('pathway')['GeneAbundance'] \
                                  .transform(lambda x: x / (x.sum() + EPS))

        g.edges['PG'].data['weight'] = torch.tensor(
            df_pg['weight_pg'].values,
            dtype=torch.float32
        )

    # =====================================================
    # MM / PP: 通常是相似度图或共现图，不做重算
    # =====================================================

    return g

# ======================================================
# 在 pruning 后的每个图中重新计算边的权重
# ======================================================
for idx, (g, meta) in enumerate(tqdm(zip(filtered_graphs, filtered_meta), total=len(filtered_graphs))):
    g = recompute_edge_weights_from_feat(g)  # 在 pruning 后重新计算边权重
    filtered_graphs[idx] = g  # 更新图

# ======================================================
# 6️⃣ 保存
# ======================================================
dgl.save_graphs(
    str(cfg.GRAPH_PRUNED_BIN),
    filtered_graphs,
    labels={"labels": all_labels}
)

with open(cfg.GRAPH_PRUNED_META_PKL, "wb") as f:
    pickle.dump(filtered_meta, f)
print("\n✅ pruning 完成：gene / pathway 丰度 > 1e-4，并且重新计算了边权重")


原始样本数: 237


  0%|          | 0/237 [00:00<?, ?it/s]

[Graph 0] gene: 518, microbe: 80, pathway: 122


 12%|█▏        | 28/237 [00:00<00:01, 141.10it/s]

[Graph 20] gene: 482, microbe: 64, pathway: 106


...

[Graph 220] gene: 509, microbe: 44, pathway: 110


100%|██████████| 237/237 [00:02<00:00, 89.58it/s] 



===== 节点数量统计（筛选后） =====
gene       | mean= 513.2 | median= 511.0 | min= 198 | max= 590
microbe    | mean=  75.9 | median=  75.0 | min=  27 | max= 140
pathway    | mean= 126.2 | median= 119.0 | min=  16 | max= 232


100%|██████████| 237/237 [00:44<00:00,  5.35it/s]



✅ pruning 完成：gene / pathway 丰度 > 1e-4，并且重新计算了边权重


In [30]:
import dgl
import torch
import pickle
from tqdm import tqdm

# ======================================================
# 1️⃣ 读取数据
# ======================================================
graphs, label_dict = dgl.load_graphs(str(cfg.GRAPH_PRUNED_BIN))
labels = label_dict["labels"]

with open(cfg.GRAPH_PRUNED_META_PKL, "rb") as f:
    metas = pickle.load(f)

print(f"Graphs: {len(graphs)}")
print(f"Labels: {labels.shape}")

# ======================================================
# 2️⃣ 全局集合（去重核心）
# ======================================================
all_microbes = set()
all_genes = set()
all_pathways = set()

# ======================================================
# 3️⃣ 遍历所有样本
# ======================================================
for g, meta in tqdm(zip(graphs, metas), total=len(graphs)):

    # -------- microbe --------
    if "microbe" in meta:
        all_microbes.update(meta["microbe"])

    # -------- gene --------
    if "gene" in meta:
        all_genes.update(meta["gene"])

    # -------- pathway --------
    if "pathway" in meta:
        all_pathways.update(meta["pathway"])


# ======================================================
# 4️⃣ 排序（保证稳定性，可复现）
# ======================================================
all_microbes = sorted(all_microbes)
all_genes = sorted(all_genes)
all_pathways = sorted(all_pathways)

# ======================================================
# 5️⃣ 构建 ID 映射（论文可用）
# ======================================================
microbe_id = {m: i for i, m in enumerate(all_microbes)}
gene_id = {g: i for i, g in enumerate(all_genes)}
pathway_id = {p: i for i, p in enumerate(all_pathways)}

# ======================================================
# 6️⃣ 输出统计信息（论文直接用）
# ======================================================
print("\n===== Dataset Feature Statistics =====")
print(f"Unique microbes : {len(all_microbes)}")
print(f"Unique genes    : {len(all_genes)}")
print(f"Unique pathways : {len(all_pathways)}")

# ======================================================
# 7️⃣ 保存（用于后续模型 or reproducibility）
# ======================================================
with open(cfg.FEATURE_VOCAB_PKL, "wb") as f:
    pickle.dump({
        "microbes": all_microbes,
        "genes": all_genes,
        "pathways": all_pathways,
        "microbe_id": microbe_id,
        "gene_id": gene_id,
        "pathway_id": pathway_id
    }, f)

print("✅ Feature vocabulary saved!")


Graphs: 237
Labels: torch.Size([237])


100%|██████████| 237/237 [00:00<00:00, 24913.54it/s]


===== Dataset Feature Statistics =====
Unique microbes : 532
Unique genes    : 591
Unique pathways : 293
✅ Feature vocabulary saved!
